In [ ]:
import os
import pandas as pd

from src.synthetic_control import SyntheticControl
from src.process_and_diagnostics import ProcessAndDiagnostics

In [ ]:
data_dir = os.path.join('..', '..', 'data')
ml_20m_dir = os.path.join(data_dir, 'MovieLens-20M')
syn_control_dir = os.path.join(ml_20m_dir, 'synthetic_control')
dataset_name = 'MovieLens'

# Read data

In [ ]:
real_df = pd.read_csv(os.path.join(syn_control_dir, 'real.csv'), index_col=0)
LLM_df = pd.read_csv(os.path.join(syn_control_dir, 'LLM.csv'), index_col=0)
real_matrix = real_df.to_numpy()
LLM_matrix = LLM_df.to_numpy()

# Diagnostics

In [ ]:
diagnostics = ProcessAndDiagnostics(real_matrix, LLM_matrix, dataset_name=dataset_name)
diagnostics_results = diagnostics.process_and_diagnostic(
    max_rank=11, 
    rank_step=2, 
    holdout_fraction=0.2, 
    check_stacked=True, 
    normalize_by_rank=True,
    fig=True,
    verbose=True
)

# Synthetic control

In [ ]:
sc = SyntheticControl(
    real_matrix,  
    LLM_matrix,  
    dataset_name=dataset_name,
    imputation_rank=5, 
    min_col_std=1
)

In [ ]:
# Column-wise evaluation with linear regression
train_mse_thresholds_col = [round(x * 0.01, 2) for x in range(10, 66)]
res_col = sc.evaluate_all_columns(
    method="linear_regression_l2", 
    regularization_multiplier=1e3, 
    train_mse_thresholds=train_mse_thresholds_col,
    n_jobs=-2,  
    verbose=True
)

In [ ]:
# Column-wise evaluation with neural network
train_mse_thresholds_col = [round(x * 0.01, 2) for x in range(10, 66)]
res_col_nn = sc.evaluate_all_columns(
    method="neural_net", 
    nn_hidden_dims=[16],
    nn_epochs=200,
    nn_lr=1e-3,
    nn_weight_decay=1e-1,
    nn_batch_size=128,
    nn_patience=20,
    nn_device="auto",
    nn_seed=42,
    train_mse_thresholds=train_mse_thresholds_col,
    n_jobs=-2,  
    verbose=True
)

In [ ]:
# Row-wise evaluation
train_mse_thresholds_row = [round(x * 0.01, 2) for x in range(20, 101)]
res_row = sc.evaluate_all_rows(
    method="linear_regression_l2", 
    regularization_multiplier=1e3, 
    train_mse_thresholds=train_mse_thresholds_row,
    n_jobs=-2,  
    verbose=True
)

In [ ]:
# Row-wise evaluation
train_mse_thresholds_row = [round(x * 0.01, 2) for x in range(20, 101)]
res_row_nn = sc.evaluate_all_rows(
    method="neural_net", 
    nn_hidden_dims=[16],
    nn_epochs=200,
    nn_lr=1e-3,
    nn_weight_decay=1,
    nn_batch_size=128,
    nn_patience=20,
    nn_device="auto",
    nn_seed=42,
    train_mse_thresholds=train_mse_thresholds_row,
    n_jobs=-2,  
    verbose=True
)

# Comparison with an additional baseline
The additional baseline is prompting LLM with in-context examples of the user's ratings for all other movies. Only used for predicing columns.

In [ ]:
real_df_in_context = pd.read_csv(os.path.join(syn_control_dir, 'real.csv'), index_col=0)
LLM_df_additional_baseline = pd.read_csv(os.path.join(syn_control_dir, 'LLM_in_context_other_ratings_full.csv'), index_col=0)
LLM_df_in_context = LLM_df.copy()
real_matrix_in_context = real_df_in_context.to_numpy()
LLM_matrix_additional_baseline = LLM_df_additional_baseline.to_numpy()
LLM_matrix_in_context = LLM_df_in_context.to_numpy()

In [ ]:
sc_additional_baseline = SyntheticControl(
    real_matrix_in_context,  
    LLM_matrix_in_context,  
    dataset_name=dataset_name + '_with_additional_baseline',
    additional_baseline_matrix=LLM_matrix_additional_baseline,
    imputation_rank=5, 
    min_col_std=1
)

In [ ]:
# Column-wise evaluation
train_mse_thresholds_col = [round(x * 0.01, 2) for x in range(10, 66)]
res_col_additional_baseline = sc_additional_baseline.evaluate_all_columns(
    method="linear_regression_l2", 
    regularization_multiplier=1e3, 
    train_mse_thresholds=train_mse_thresholds_col,
    n_jobs=-2,  
    verbose=True
)